# BERT / RoBERTa fine-tuning on English-translated tweets: schedulers, Optuna search and contextual augmentation

In [ ]:
DATA_DIR = "../data"

In [2]:
import pandas as pd
train_eng_df = pd.read_csv(f'{DATA_DIR}/train_eng_v2.csv')

In [3]:
train_eng_df.label.value_counts()

label
Politik                    2858
Sosial Budaya               381
Ekonomi                     263
Pertahanan dan Keamanan     255
Ideologi                    253
Sumber Daya Alam            134
Demografi                    55
Geografi                     19
Name: count, dtype: int64

In [1]:
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split

# Load the CSV file
file_path = f'{DATA_DIR}/train_eng_v2.csv'
df = pd.read_csv(file_path)

# Create a label mapping
label_mapping = {
    'Politik': 0,
    'Sosial Budaya': 1,
    'Ekonomi': 2,
    'Pertahanan dan Keamanan': 3,
    'Ideologi': 4,
    'Sumber Daya Alam': 5,
    'Demografi': 6,
    'Geografi': 7
}

# Apply the label mapping
df['label'] = df['label'].map(label_mapping)

# Split the DataFrame into training and validation sets
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

# Convert DataFrames to Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Combine into a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

# Tokenize the dataset
model_name = "cardiffnlp/roberta-large-tweet-topic-single-all"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)

Map:   0%|          | 0/3374 [00:00<?, ? examples/s]

Map:   0%|          | 0/844 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/roberta-large-tweet-topic-single-all and are newly initialized because the shapes did not match:
- classifier.out_proj.weight: found shape torch.Size([6, 1024]) in the checkpoint and torch.Size([8, 1024]) in the model instantiated
- classifier.out_proj.bias: found shape torch.Size([6]) in the checkpoint and torch.Size([8]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [71]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=7,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)


# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/2954 [00:00<?, ?it/s]

  0%|          | 0/106 [00:00<?, ?it/s]

{'eval_loss': 0.8514370918273926, 'eval_runtime': 6.2076, 'eval_samples_per_second': 135.962, 'eval_steps_per_second': 17.076, 'epoch': 1.0}
{'loss': 0.9354, 'grad_norm': 20.451250076293945, 'learning_rate': 1.6635071090047395e-05, 'epoch': 1.18}


  0%|          | 0/106 [00:00<?, ?it/s]

...

{'eval_loss': 1.5495885610580444, 'eval_runtime': 6.2687, 'eval_samples_per_second': 134.637, 'eval_steps_per_second': 16.909, 'epoch': 6.0}


  0%|          | 0/106 [00:00<?, ?it/s]

{'eval_loss': 1.6634888648986816, 'eval_runtime': 6.3301, 'eval_samples_per_second': 133.331, 'eval_steps_per_second': 16.745, 'epoch': 7.0}
{'train_runtime': 998.3059, 'train_samples_per_second': 23.658, 'train_steps_per_second': 2.959, 'train_loss': 0.46432443849929056, 'epoch': 7.0}


  0%|          | 0/106 [00:00<?, ?it/s]

Balanced Accuracy: 0.40651244219830296


In [72]:
# Define training arguments with cosine scheduler
training_args = TrainingArguments(
    output_dir="./cosine",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    learning_rate=2e-5,
    per_device_train_batch_size=8,  # Use a smaller batch size
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    lr_scheduler_type="cosine",  # Use cosine scheduler
    warmup_steps=500,
    logging_dir='./logs',
    logging_steps=10,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)


# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/1266 [00:00<?, ?it/s]

{'loss': 0.5433, 'grad_norm': 13.549372673034668, 'learning_rate': 3.6e-07, 'epoch': 0.02}
{'loss': 0.6175, 'grad_norm': 8.441020965576172, 'learning_rate': 7.2e-07, 'epoch': 0.05}
{'loss': 0.3655, 'grad_norm': 10.053191184997559, 'learning_rate': 1.12e-06, 'epoch': 0.07}
{'loss': 0.3727, 'grad_norm': 32.589027404785156, 'learning_rate': 1.52e-06, 'epoch': 0.09}
{'loss': 0.6547, 'grad_norm': 19.343053817749023, 'learning_rate': 1.9200000000000003e-06, 'epoch': 0.12}
{'loss': 0.5346, 'grad_norm': 4.255354881286621, 'learning_rate': 2.3200000000000002e-06, 'epoch': 0.14}
{'loss': 0.5485, 'grad_norm': 48.836002349853516, 'learning_rate': 2.7200000000000002e-06, 'epoch': 0.17}
{'loss': 0.4446, 'grad_norm': 8.987371444702148, 'learning_rate': 3.08e-06, 'epoch': 0.19}
{'loss': 0.4276, 'grad_norm': 23.138946533203125, 'learning_rate': 3.48e-06, 'epoch': 0.21}
{'loss': 0.423, 'grad_norm': 2.7648346424102783, 'learning_rate': 3.88e-06, 'epoch': 0.24}
{'loss': 0.3518, 'grad_norm': 8.923154830932

  0%|          | 0/106 [00:00<?, ?it/s]

{'eval_loss': 1.0251097679138184, 'eval_runtime': 6.2544, 'eval_samples_per_second': 134.945, 'eval_steps_per_second': 16.948, 'epoch': 1.0}
{'loss': 0.2629, 'grad_norm': 40.76283264160156, 'learning_rate': 1.704e-05, 'epoch': 1.02}
{'loss': 0.3723, 'grad_norm': 67.05563354492188, 'learning_rate': 1.7440000000000002e-05, 'epoch': 1.04}
{'loss': 0.5947, 'grad_norm': 33.317386627197266, 'learning_rate': 1.7840000000000002e-05, 'epoch': 1.07}
{'loss': 0.3243, 'grad_norm': 28.078092575073242, 'learning_rate': 1.824e-05, 'epoch': 1.09}
{'loss': 0.4782, 'grad_norm': 5.5779900550842285, 'learning_rate': 1.86e-05, 'epoch': 1.11}
{'loss': 0.4271, 'grad_norm': 29.599376678466797, 'learning_rate': 1.9e-05, 'epoch': 1.14}
{'loss': 0.4461, 'grad_norm': 39.15562057495117, 'learning_rate': 1.94e-05, 'epoch': 1.16}
{'loss': 0.3575, 'grad_norm': 26.252456665039062, 'learning_rate': 1.98e-05, 'epoch': 1.18}
{'loss': 0.759, 'grad_norm': 13.7485990524292, 'learning_rate': 1.9997897495179932e-05, 'epoch': 

...

{'eval_loss': 1.2015122175216675, 'eval_runtime': 6.3277, 'eval_samples_per_second': 133.382, 'eval_steps_per_second': 16.752, 'epoch': 2.0}
{'loss': 0.1884, 'grad_norm': 50.39581298828125, 'learning_rate': 1.1592693779646405e-05, 'epoch': 2.01}
{'loss': 0.4636, 'grad_norm': 5.622215270996094, 'learning_rate': 1.1186573610485099e-05, 'epoch': 2.04}
{'loss': 0.3982, 'grad_norm': 44.9615478515625, 'learning_rate': 1.0778457829746668e-05, 'epoch': 2.06}
{'loss': 0.584, 'grad_norm': 140.10926818847656, 'learning_rate': 1.036903281758456e-05, 'epoch': 2.09}
{'loss': 0.3144, 'grad_norm': 3.3102869987487793, 'learning_rate': 9.958987156053046e-06, 'epoch': 2.11}
{'loss': 0.4342, 'grad_norm': 15.877002716064453, 'learning_rate': 9.549010471031256e-06, 'epoch': 2.13}
{'loss': 0.3473, 'grad_norm': 60.529483795166016, 'learning_rate': 9.139792272391791e-06, 'epoch': 2.16}
{'loss': 0.387, 'grad_norm': 65.7422866821289, 'learning_rate': 8.732020794364327e-06, 'epoch': 2.18}
{'loss': 0.2493, 'grad_n

  0%|          | 0/106 [00:00<?, ?it/s]

{'eval_loss': 1.366496205329895, 'eval_runtime': 6.2215, 'eval_samples_per_second': 135.659, 'eval_steps_per_second': 17.038, 'epoch': 3.0}
{'train_runtime': 435.5094, 'train_samples_per_second': 23.242, 'train_steps_per_second': 2.907, 'train_loss': 0.398842003202363, 'epoch': 3.0}


  0%|          | 0/106 [00:00<?, ?it/s]

Balanced Accuracy: 0.39718867195669283


In [73]:
# Define training arguments with cosine scheduler
training_args = TrainingArguments(
    output_dir="./linear",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    learning_rate=2e-5,
    per_device_train_batch_size=8,  # Use a smaller batch size
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    lr_scheduler_type="linear",  # Use cosine scheduler
    warmup_steps=500,
    logging_dir='./logs',
    logging_steps=10,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)

/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [74]:
# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

  0%|          | 0/1266 [00:00<?, ?it/s]

{'loss': 0.597, 'grad_norm': 19.70569610595703, 'learning_rate': 3.2e-07, 'epoch': 0.02}
{'loss': 0.592, 'grad_norm': 37.580482482910156, 'learning_rate': 7.2e-07, 'epoch': 0.05}
{'loss': 0.3416, 'grad_norm': 19.73887825012207, 'learning_rate': 1.12e-06, 'epoch': 0.07}
{'loss': 0.3186, 'grad_norm': 49.040733337402344, 'learning_rate': 1.52e-06, 'epoch': 0.09}
{'loss': 0.6037, 'grad_norm': 30.002079010009766, 'learning_rate': 1.9200000000000003e-06, 'epoch': 0.12}
{'loss': 0.4517, 'grad_norm': 0.6154088377952576, 'learning_rate': 2.3200000000000002e-06, 'epoch': 0.14}
{'loss': 0.415, 'grad_norm': 42.46630096435547, 'learning_rate': 2.7200000000000002e-06, 'epoch': 0.17}
{'loss': 0.3691, 'grad_norm': 5.367023468017578, 'learning_rate': 3.12e-06, 'epoch': 0.19}
{'loss': 0.268, 'grad_norm': 25.90089225769043, 'learning_rate': 3.52e-06, 'epoch': 0.21}
{'loss': 0.2645, 'grad_norm': 0.4082046449184418, 'learning_rate': 3.920000000000001e-06, 'epoch': 0.24}
{'loss': 0.2467, 'grad_norm': 9.4088

  0%|          | 0/106 [00:00<?, ?it/s]

{'eval_loss': 1.431194543838501, 'eval_runtime': 6.3379, 'eval_samples_per_second': 133.167, 'eval_steps_per_second': 16.725, 'epoch': 1.0}
{'loss': 0.3233, 'grad_norm': 66.324462890625, 'learning_rate': 1.704e-05, 'epoch': 1.02}
{'loss': 0.2667, 'grad_norm': 104.98839569091797, 'learning_rate': 1.7440000000000002e-05, 'epoch': 1.04}
{'loss': 0.6225, 'grad_norm': 84.27437591552734, 'learning_rate': 1.7840000000000002e-05, 'epoch': 1.07}
{'loss': 0.2582, 'grad_norm': 28.592395782470703, 'learning_rate': 1.824e-05, 'epoch': 1.09}
{'loss': 0.3869, 'grad_norm': 115.58715057373047, 'learning_rate': 1.864e-05, 'epoch': 1.11}
{'loss': 0.4067, 'grad_norm': 100.38384246826172, 'learning_rate': 1.904e-05, 'epoch': 1.14}
{'loss': 0.2948, 'grad_norm': 74.11288452148438, 'learning_rate': 1.944e-05, 'epoch': 1.16}
{'loss': 0.2696, 'grad_norm': 87.64501190185547, 'learning_rate': 1.9840000000000003e-05, 'epoch': 1.18}
{'loss': 0.8311, 'grad_norm': 111.31649017333984, 'learning_rate': 1.98433420365535

  0%|          | 0/106 [00:00<?, ?it/s]

{'eval_loss': 1.3310879468917847, 'eval_runtime': 6.3371, 'eval_samples_per_second': 133.184, 'eval_steps_per_second': 16.727, 'epoch': 2.0}
{'loss': 0.2175, 'grad_norm': 3.333611011505127, 'learning_rate': 1.0966057441253265e-05, 'epoch': 2.01}
{'loss': 0.3224, 'grad_norm': 0.24859365820884705, 'learning_rate': 1.0704960835509139e-05, 'epoch': 2.04}
{'loss': 0.2518, 'grad_norm': 99.69461822509766, 'learning_rate': 1.0443864229765015e-05, 'epoch': 2.06}
{'loss': 0.6263, 'grad_norm': 155.5381317138672, 'learning_rate': 1.0182767624020888e-05, 'epoch': 2.09}
{'loss': 0.2839, 'grad_norm': 1.1282457113265991, 'learning_rate': 9.921671018276763e-06, 'epoch': 2.11}
{'loss': 0.2168, 'grad_norm': 3.824004888534546, 'learning_rate': 9.660574412532639e-06, 'epoch': 2.13}
{'loss': 0.2782, 'grad_norm': 0.8301821947097778, 'learning_rate': 9.399477806788514e-06, 'epoch': 2.16}
{'loss': 0.4057, 'grad_norm': 46.02900695800781, 'learning_rate': 9.138381201044387e-06, 'epoch': 2.18}
{'loss': 0.1885, 'g

  0%|          | 0/106 [00:00<?, ?it/s]

{'eval_loss': 1.5171339511871338, 'eval_runtime': 6.3044, 'eval_samples_per_second': 133.875, 'eval_steps_per_second': 16.814, 'epoch': 3.0}
{'train_runtime': 435.9837, 'train_samples_per_second': 23.216, 'train_steps_per_second': 2.904, 'train_loss': 0.3290456482113632, 'epoch': 3.0}


  0%|          | 0/106 [00:00<?, ?it/s]

Balanced Accuracy: 0.4377805587134403


In [3]:
# Tokenize the dataset
model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)

Map:   0%|          | 0/3374 [00:00<?, ? examples/s]

Map:   0%|          | 0/844 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# Define training arguments with cosine scheduler
training_args = TrainingArguments(
    output_dir="./linearRoberta",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    learning_rate=2e-5,
    per_device_train_batch_size=32,  # Use a smaller batch size
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    lr_scheduler_type="linear",  # Use cosine scheduler
    warmup_steps=500,
    logging_dir='./logs',
    logging_steps=10,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)


# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/318 [00:00<?, ?it/s]

{'loss': 1.9691, 'grad_norm': 6.582851409912109, 'learning_rate': 4.0000000000000003e-07, 'epoch': 0.09}
{'loss': 1.952, 'grad_norm': 5.6855082511901855, 'learning_rate': 8.000000000000001e-07, 'epoch': 0.19}
{'loss': 1.9579, 'grad_norm': 5.854674816131592, 'learning_rate': 1.2000000000000002e-06, 'epoch': 0.28}
{'loss': 1.8995, 'grad_norm': 6.502702236175537, 'learning_rate': 1.6000000000000001e-06, 'epoch': 0.38}
{'loss': 1.8717, 'grad_norm': 5.269619941711426, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.47}
{'loss': 1.8073, 'grad_norm': 4.189080238342285, 'learning_rate': 2.4000000000000003e-06, 'epoch': 0.57}
{'loss': 1.7072, 'grad_norm': 6.925071716308594, 'learning_rate': 2.8000000000000003e-06, 'epoch': 0.66}
{'loss': 1.5445, 'grad_norm': 7.932063102722168, 'learning_rate': 3.2000000000000003e-06, 'epoch': 0.75}
{'loss': 1.3102, 'grad_norm': 7.009544372558594, 'learning_rate': 3.6000000000000003e-06, 'epoch': 0.85}
{'loss': 1.1538, 'grad_norm': 3.4813458919525146, 'learn

  0%|          | 0/27 [00:00<?, ?it/s]

{'eval_loss': 1.1820622682571411, 'eval_runtime': 4.7402, 'eval_samples_per_second': 178.052, 'eval_steps_per_second': 5.696, 'epoch': 1.0}
{'loss': 1.2259, 'grad_norm': 3.4648213386535645, 'learning_rate': 4.4e-06, 'epoch': 1.04}
{'loss': 1.2036, 'grad_norm': 4.418182849884033, 'learning_rate': 4.800000000000001e-06, 'epoch': 1.13}
{'loss': 1.2106, 'grad_norm': 4.172192573547363, 'learning_rate': 5.2e-06, 'epoch': 1.23}
{'loss': 1.1154, 'grad_norm': 4.2575154304504395, 'learning_rate': 5.600000000000001e-06, 'epoch': 1.32}
{'loss': 1.1125, 'grad_norm': 4.374039649963379, 'learning_rate': 6e-06, 'epoch': 1.42}
{'loss': 1.0806, 'grad_norm': 4.394668102264404, 'learning_rate': 6.4000000000000006e-06, 'epoch': 1.51}
{'loss': 1.1727, 'grad_norm': 6.111232280731201, 'learning_rate': 6.800000000000001e-06, 'epoch': 1.6}
{'loss': 0.9637, 'grad_norm': 5.012210369110107, 'learning_rate': 7.2000000000000005e-06, 'epoch': 1.7}
{'loss': 1.0041, 'grad_norm': 6.990807056427002, 'learning_rate': 7.60

...

{'eval_loss': 0.9748303294181824, 'eval_runtime': 4.7704, 'eval_samples_per_second': 176.925, 'eval_steps_per_second': 5.66, 'epoch': 2.0}
{'loss': 0.9359, 'grad_norm': 4.680797100067139, 'learning_rate': 8.8e-06, 'epoch': 2.08}
{'loss': 0.9008, 'grad_norm': 6.928817272186279, 'learning_rate': 9.200000000000002e-06, 'epoch': 2.17}
{'loss': 0.9228, 'grad_norm': 13.425155639648438, 'learning_rate': 9.600000000000001e-06, 'epoch': 2.26}
{'loss': 0.8582, 'grad_norm': 8.33742904663086, 'learning_rate': 1e-05, 'epoch': 2.36}
{'loss': 0.9631, 'grad_norm': 13.774367332458496, 'learning_rate': 1.04e-05, 'epoch': 2.45}
{'loss': 0.9402, 'grad_norm': 11.575005531311035, 'learning_rate': 1.0800000000000002e-05, 'epoch': 2.55}
{'loss': 0.8572, 'grad_norm': 19.969196319580078, 'learning_rate': 1.1200000000000001e-05, 'epoch': 2.64}
{'loss': 0.8795, 'grad_norm': 8.901304244995117, 'learning_rate': 1.16e-05, 'epoch': 2.74}
{'loss': 0.7947, 'grad_norm': 10.369805335998535, 'learning_rate': 1.2e-05, 'epo

  0%|          | 0/27 [00:00<?, ?it/s]

{'eval_loss': 0.8384498357772827, 'eval_runtime': 4.7694, 'eval_samples_per_second': 176.962, 'eval_steps_per_second': 5.661, 'epoch': 3.0}
{'train_runtime': 207.4917, 'train_samples_per_second': 48.783, 'train_steps_per_second': 1.533, 'train_loss': 1.2221669850859251, 'epoch': 3.0}


  0%|          | 0/27 [00:00<?, ?it/s]

Balanced Accuracy: 0.32985425223384507


In [76]:
training_args = TrainingArguments(
    output_dir="./roberta",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=7,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)


# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/742 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

{'eval_loss': 0.9491597414016724, 'eval_runtime': 2.453, 'eval_samples_per_second': 344.069, 'eval_steps_per_second': 11.007, 'epoch': 1.0}


  0%|          | 0/27 [00:00<?, ?it/s]

...

{'eval_loss': 0.8475618958473206, 'eval_runtime': 2.4463, 'eval_samples_per_second': 345.01, 'eval_steps_per_second': 11.037, 'epoch': 6.0}


  0%|          | 0/27 [00:00<?, ?it/s]

{'eval_loss': 0.8558804988861084, 'eval_runtime': 2.3892, 'eval_samples_per_second': 353.252, 'eval_steps_per_second': 11.301, 'epoch': 7.0}
{'train_runtime': 334.1379, 'train_samples_per_second': 70.683, 'train_steps_per_second': 2.221, 'train_loss': 0.6260320658311047, 'epoch': 7.0}


  0%|          | 0/27 [00:00<?, ?it/s]

Balanced Accuracy: 0.39755114307636363


In [77]:
training_args = TrainingArguments(
    output_dir="./roberta3",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)


# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/318 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

{'eval_loss': 0.8476194143295288, 'eval_runtime': 2.3843, 'eval_samples_per_second': 353.987, 'eval_steps_per_second': 11.324, 'epoch': 1.0}


  0%|          | 0/27 [00:00<?, ?it/s]

{'eval_loss': 0.8968831300735474, 'eval_runtime': 2.3067, 'eval_samples_per_second': 365.89, 'eval_steps_per_second': 11.705, 'epoch': 2.0}


  0%|          | 0/27 [00:00<?, ?it/s]

{'eval_loss': 0.9237827062606812, 'eval_runtime': 2.4462, 'eval_samples_per_second': 345.029, 'eval_steps_per_second': 11.038, 'epoch': 3.0}
{'train_runtime': 140.698, 'train_samples_per_second': 71.941, 'train_steps_per_second': 2.26, 'train_loss': 0.39516446575428704, 'epoch': 3.0}


  0%|          | 0/27 [00:00<?, ?it/s]

Balanced Accuracy: 0.4501072646720757


In [5]:
# Tokenize the dataset
model_name = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)

Map:   0%|          | 0/3374 [00:00<?, ? examples/s]

Map:   0%|          | 0/844 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args = TrainingArguments(
    output_dir="./bert3",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)


# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

In [6]:
training_args = TrainingArguments(
    output_dir="./bert",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=7,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)


# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/371 [00:00<?, ?it/s]

  0%|          | 0/14 [00:00<?, ?it/s]

{'eval_loss': 1.0882431268692017, 'eval_runtime': 3.1895, 'eval_samples_per_second': 264.618, 'eval_steps_per_second': 4.389, 'epoch': 1.0}


  0%|          | 0/14 [00:00<?, ?it/s]

...

{'eval_loss': 0.9282215237617493, 'eval_runtime': 3.1387, 'eval_samples_per_second': 268.898, 'eval_steps_per_second': 4.46, 'epoch': 6.0}


  0%|          | 0/14 [00:00<?, ?it/s]

{'eval_loss': 0.9276850819587708, 'eval_runtime': 3.1785, 'eval_samples_per_second': 265.536, 'eval_steps_per_second': 4.405, 'epoch': 7.0}
{'train_runtime': 357.3766, 'train_samples_per_second': 66.087, 'train_steps_per_second': 1.038, 'train_loss': 0.7913979584316038, 'epoch': 7.0}


  0%|          | 0/14 [00:00<?, ?it/s]

Balanced Accuracy: 0.34046317452875957


In [7]:
training_args = TrainingArguments(
    output_dir="./bert3",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)


# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/159 [00:00<?, ?it/s]

  0%|          | 0/14 [00:00<?, ?it/s]

{'eval_loss': 0.9466595649719238, 'eval_runtime': 3.129, 'eval_samples_per_second': 269.734, 'eval_steps_per_second': 4.474, 'epoch': 1.0}


  0%|          | 0/14 [00:00<?, ?it/s]

{'eval_loss': 0.9691168069839478, 'eval_runtime': 3.2148, 'eval_samples_per_second': 262.537, 'eval_steps_per_second': 4.355, 'epoch': 2.0}


  0%|          | 0/14 [00:00<?, ?it/s]

{'eval_loss': 0.9971573948860168, 'eval_runtime': 3.1651, 'eval_samples_per_second': 266.661, 'eval_steps_per_second': 4.423, 'epoch': 3.0}
{'train_runtime': 153.9716, 'train_samples_per_second': 65.739, 'train_steps_per_second': 1.033, 'train_loss': 0.44172625271779187, 'epoch': 3.0}


  0%|          | 0/14 [00:00<?, ?it/s]

Balanced Accuracy: 0.33307720221441817


In [1]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import balanced_accuracy_score
import optuna
import torch
import gc

# Load the CSV file
file_path = f'{DATA_DIR}/train_eng_v2.csv'
df = pd.read_csv(file_path)

# Create a label mapping
label_mapping = {
    'Politik': 0,
    'Sosial Budaya': 1,
    'Ekonomi': 2,
    'Pertahanan dan Keamanan': 3,
    'Ideologi': 4,
    'Sumber Daya Alam': 5,
    'Demografi': 6,
    'Geografi': 7
}

# Apply the label mapping
df['label'] = df['label'].map(label_mapping)

# Split the DataFrame into training and validation sets
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

# Convert DataFrames to Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Combine into a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

# Tokenize the dataset
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Define the objective function for Optuna
def objective(trial):
    # Hyperparameters to tune
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-8, 1e-4)
    batch_size = trial.suggest_categorical('batch_size', [8, 16, 32])
    num_train_epochs = trial.suggest_int('num_train_epochs', 2, 10)

    # Clear GPU memory
    torch.cuda.empty_cache()
    gc.collect()

    # Define the model
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)

    # Define training arguments
    training_args = TrainingArguments(
        output_dir="./results",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )

    # Define Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets['train'],
        eval_dataset=tokenized_datasets['validation'],
    )

    # Train the model
    trainer.train()

    # Evaluate the model
    eval_result = trainer.evaluate()
    
    # Return the evaluation metric to be optimized
    return eval_result['eval_loss']

# Callback to save the best trial
def save_best_trial(study, trial):
    if study.best_trial == trial:
        with open("best_hyperparameters.txt", "w") as f:
            f.write(str(study.best_params))

# Create a study and optimize the objective function
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20, callbacks=[save_best_trial])

# Get the best hyperparameters
best_hyperparameters = study.best_params
print(best_hyperparameters)

# Save the best hyperparameters to a file
with open("best_hyperparameters.txt", "w") as f:
    f.write(str(best_hyperparameters))

# Clear GPU memory before final training
torch.cuda.empty_cache()
gc.collect()

# Train and evaluate the model with the best hyperparameters
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)

# Define training arguments with the best hyperparameters
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=best_hyperparameters['learning_rate'],
    per_device_train_batch_size=best_hyperparameters['batch_size'],
    per_device_eval_batch_size=best_hyperparameters['batch_size'],
    num_train_epochs=best_hyperparameters['num_train_epochs'],
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer with the best hyperparameters
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)

# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

Map:   0%|          | 0/3374 [00:00<?, ? examples/s]

Map:   0%|          | 0/844 [00:00<?, ? examples/s]

[I 2024-06-02 15:03:30,640] A new study created in memory with name: no-name-2614de97-f129-4da5-bee8-14bcecd207be
/tmp/ipykernel_3223573/1514148416.py:54: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-8, 1e-4)


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

{'eval_loss': 1.6054861545562744, 'eval_runtime': 4.7358, 'eval_samples_per_second': 178.215, 'eval_steps_per_second': 11.191, 'epoch': 4.0}
{'loss': 1.6675, 'grad_norm': 16.90633773803711, 'learning_rate': 8.669647493282008e-08, 'epoch': 4.74}


  0%|          | 0/53 [00:00<?, ?it/s]

{'eval_loss': 1.327257752418518, 'eval_runtime': 4.5711, 'eval_samples_per_second': 184.637, 'eval_steps_per_second': 11.594, 'epoch': 5.0}


[W 2024-06-02 17:00:41,124] Trial 17 failed with parameters: {'learning_rate': 1.6480140730473007e-07, 'batch_size': 16, 'num_train_epochs': 10} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/optuna/study/_optimize.py", line 196, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_3223573/1514148416.py", line 89, in objective
    trainer.train()
  File "/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/trainer.py", line 1885, in train
    return inner_training_loop(
  File "/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/trainer.py", line 2218, in _inner_training_loop
    if (
KeyboardInterrupt
[W 2024-06-02 17:00:41,126] Trial 17 failed with value None.


KeyboardInterrupt: 

In [12]:
# clear memory with torch
import torch
torch.cuda.empty_cache()
gc.collect()

0

In [2]:
import ast
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification
from sklearn.metrics import balanced_accuracy_score

# Load best hyperparameters from a text file
with open("best_hyperparameters.txt", "r") as f:
    best_hyperparameters = ast.literal_eval(f.read())

# Load the model
model_name = "roberta-base"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)

# Define training arguments with the best hyperparameters
training_args = TrainingArguments(
    output_dir="./roberta-best-param",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=best_hyperparameters['learning_rate'],
    per_device_train_batch_size=best_hyperparameters['batch_size'],
    per_device_eval_batch_size=best_hyperparameters['batch_size'],
    num_train_epochs=best_hyperparameters['num_train_epochs'],
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer with the best hyperparameters
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)

# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/1477 [00:00<?, ?it/s]

  0%|          | 0/53 [00:00<?, ?it/s]

{'eval_loss': 0.9475564956665039, 'eval_runtime': 4.654, 'eval_samples_per_second': 181.35, 'eval_steps_per_second': 11.388, 'epoch': 1.0}


  0%|          | 0/53 [00:00<?, ?it/s]

...

{'eval_loss': 0.7978417277336121, 'eval_runtime': 4.7975, 'eval_samples_per_second': 175.924, 'eval_steps_per_second': 11.047, 'epoch': 6.0}


  0%|          | 0/53 [00:00<?, ?it/s]

{'eval_loss': 0.8045294284820557, 'eval_runtime': 4.4659, 'eval_samples_per_second': 188.989, 'eval_steps_per_second': 11.868, 'epoch': 7.0}
{'train_runtime': 492.1352, 'train_samples_per_second': 47.991, 'train_steps_per_second': 3.001, 'train_loss': 0.7444559457581351, 'epoch': 7.0}


  0%|          | 0/53 [00:00<?, ?it/s]

Balanced Accuracy: 0.41636881143284843


In [12]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import balanced_accuracy_score
import nlpaug.augmenter.word as naw
import ast

# Load the CSV file
file_path = f'{DATA_DIR}/train_eng_v2.csv'
df = pd.read_csv(file_path)

# Create a label mapping
label_mapping = {
    'Politik': 0,
    'Sosial Budaya': 1,
    'Ekonomi': 2,
    'Pertahanan dan Keamanan': 3,
    'Ideologi': 4,
    'Sumber Daya Alam': 5,
    'Demografi': 6,
    'Geografi': 7
}

# Apply the label mapping
df['label'] = df['label'].map(label_mapping)

# Split the DataFrame into training and validation sets
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'])

# Create an augmenter using BERT
aug = naw.ContextualWordEmbsAug(model_path='bert-base-uncased', action="substitute")

# Function to augment data
def augment_data(df, target_count):
    augmented_texts = []
    augmented_labels = []
    
    for label in df['label'].unique():
        subset = df[df['label'] == label]
        current_count = len(subset)
        
        while current_count < target_count:
            for text in subset['text']:
                augmented_text = aug.augment(text)[0]  # Take the string out of the list
                augmented_texts.append(augmented_text)
                augmented_labels.append(label)
                current_count += 1
                if current_count >= target_count:
                    break

    augmented_df = pd.DataFrame({'text': augmented_texts, 'label': augmented_labels})
    return pd.concat([df, augmented_df], ignore_index=True)

# Augment the training data to ensure each class has at least 305 examples
train_df_augmented = augment_data(train_df, 305)

In [13]:
train_df_augmented['label'].value_counts()

label
0    2286
3     305
4     305
2     305
1     305
6     305
5     305
7     305
Name: count, dtype: int64

In [15]:
train_df_augmented.to_csv(f"{DATA_DIR}/trainaug.csv", index=False)

In [16]:
val_df.to_csv(f"{DATA_DIR}/val.csv", index=False)

## AUGMENTATION BELOW

In [4]:
import re
from nltk.corpus import stopwords
import pandas as pd


def preprocess_text(text):
    # Define stopwords
    stop_words = set(stopwords.words('english'))
    # Add custom words to remove
    custom_words = {'anies', 'prabowo', 'gibran', 'ganjar'}
    all_stopwords = stop_words.union(custom_words)

    # Remove hashtags
    text = re.sub(r'#\w+', '', text)
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    # Remove stopwords and custom words
    text = ' '.join([word for word in text.split() if word.lower() not in all_stopwords])

    return text

train_df_augmented = pd.read_csv(f"{DATA_DIR}/trainaug.csv")
val_df = pd.read_csv(f"{DATA_DIR}/val.csv")

In [5]:
# Convert DataFrames to Datasets
train_dataset = Dataset.from_pandas(train_df_augmented)
val_dataset = Dataset.from_pandas(val_df)

# Combine into a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

# Tokenize the dataset
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./xlmroberta3augnostop",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)

# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

Map:   0%|          | 0/4421 [00:00<?, ? examples/s]

Map:   0%|          | 0/844 [00:00<?, ? examples/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/831 [00:00<?, ?it/s]

  0%|          | 0/53 [00:00<?, ?it/s]

...

{'eval_loss': 0.8297074437141418, 'eval_runtime': 4.6986, 'eval_samples_per_second': 179.628, 'eval_steps_per_second': 11.28, 'epoch': 2.0}


  0%|          | 0/53 [00:00<?, ?it/s]

{'eval_loss': 0.8164283037185669, 'eval_runtime': 4.337, 'eval_samples_per_second': 194.606, 'eval_steps_per_second': 12.221, 'epoch': 3.0}
{'train_runtime': 330.0393, 'train_samples_per_second': 40.186, 'train_steps_per_second': 2.518, 'train_loss': 1.0019352037266847, 'epoch': 3.0}


  0%|          | 0/53 [00:00<?, ?it/s]

Balanced Accuracy: 0.3866808154200817


In [6]:
train_df_augmented

,text,label
0,Law No. 16 of 2012 concerning the Defense Indu...,3
1,Ideally the second round would be Ganjar - Mah...,0
2,KN 02 Prabowo Gibran row of KPK patients. Khof...,0
3,JK on Purchasing Used Defense Equipment from t...,3
4,"Nobar with my mother for treatment, the commen...",0
...,...,...
4416,so no want to move to ikn later if i wish my l...,7
4417,"my village lives in ruins, near north high hil...",7
4418,"remember my supporters ; lift your spirits up,...",7
4419,"new year's eve, reward salawat & wednesday ; i...",7


In [17]:
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split

train_df_augmented = pd.read_csv(f"{DATA_DIR}/trainaug.csv")
val_df = pd.read_csv(f"{DATA_DIR}/val.csv")

# Convert DataFrames to Datasets
train_dataset = Dataset.from_pandas(train_df_augmented)
val_dataset = Dataset.from_pandas(val_df)

# Combine into a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

# Tokenize the dataset
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)


training_args = TrainingArguments(
    output_dir="./roberta3aug",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)


# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

Map:   0%|          | 0/4421 [00:00<?, ? examples/s]

Map:   0%|          | 0/844 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/417 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

...

{'eval_loss': 0.8069957494735718, 'eval_runtime': 4.147, 'eval_samples_per_second': 203.523, 'eval_steps_per_second': 6.511, 'epoch': 2.0}


  0%|          | 0/27 [00:00<?, ?it/s]

{'eval_loss': 0.8015658259391785, 'eval_runtime': 4.335, 'eval_samples_per_second': 194.694, 'eval_steps_per_second': 6.228, 'epoch': 3.0}
{'train_runtime': 241.9996, 'train_samples_per_second': 54.806, 'train_steps_per_second': 1.723, 'train_loss': 0.9027446499831385, 'epoch': 3.0}


  0%|          | 0/27 [00:00<?, ?it/s]

Balanced Accuracy: 0.4242008959587381


In [1]:
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
import nltk
import re
from nltk.corpus import stopwords

# Download NLTK stopwords
nltk.download('stopwords')

def preprocess_text(text):
    # Define stopwords
    stop_words = set(stopwords.words('english'))
    # Add custom words to remove
    custom_words = {'imin','anies', 'prabowo', 'gibran', 'ganjar', 'jokowi', 'sandiaga', 'ahok','mahfud', 'machfud'}
    all_stopwords = stop_words.union(custom_words)

    # Remove stopwords and custom words
    text = ' '.join([word for word in text.split() if word.lower() not in all_stopwords])

    return text

# Load the CSV files
train_df_augmented = pd.read_csv(f"{DATA_DIR}/trainaug.csv")
val_df = pd.read_csv(f"{DATA_DIR}/val.csv")

# Apply preprocessing to the text column
train_df_augmented['text'] = train_df_augmented['text'].apply(preprocess_text)
val_df['text'] = val_df['text'].apply(preprocess_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/ikhlasul.hanif/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
train_df_augmented

,text,label
0,Law No. 16 2012 concerning Defense Industry in...,3
1,Ideally second round would - vs - Cak think on...,0
2,KN 02 row KPK patients. Khofifah grant funds. ...,0
3,JK Purchasing Used Defense Equipment Era: 25 Y...,3
4,"Nobar mother treatment, comment ""Why mean teas...",0
...,...,...
4416,want move ikn later wish luck. hopefully futur...,7
4417,"village lives ruins, near north high hill. far...",7
4418,"remember supporters ; lift spirits up, soon ev...",7
4419,"new year's eve, reward salawat & wednesday ; i...",7


In [2]:
# Convert DataFrames to Datasets
train_dataset = Dataset.from_pandas(train_df_augmented)
val_dataset = Dataset.from_pandas(val_df)

# Combine into a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

# Tokenize the dataset
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./roberta3augnostop",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)

# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

Map:   0%|          | 0/4421 [00:00<?, ? examples/s]

Map:   0%|          | 0/844 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/417 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

...

{'eval_loss': 0.85716712474823, 'eval_runtime': 4.7003, 'eval_samples_per_second': 179.565, 'eval_steps_per_second': 5.744, 'epoch': 2.0}


  0%|          | 0/27 [00:00<?, ?it/s]

{'eval_loss': 0.8400566577911377, 'eval_runtime': 4.7073, 'eval_samples_per_second': 179.294, 'eval_steps_per_second': 5.736, 'epoch': 3.0}
{'train_runtime': 256.9132, 'train_samples_per_second': 51.624, 'train_steps_per_second': 1.623, 'train_loss': 1.0063824916629196, 'epoch': 3.0}


  0%|          | 0/27 [00:00<?, ?it/s]

Balanced Accuracy: 0.3929186703859582


In [3]:
# Convert DataFrames to Datasets
train_dataset = Dataset.from_pandas(train_df_augmented)
val_dataset = Dataset.from_pandas(val_df)

# Combine into a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

# Tokenize the dataset
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./xlmroberta3augnostop",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)

# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

Map:   0%|          | 0/4421 [00:00<?, ? examples/s]

Map:   0%|          | 0/844 [00:00<?, ? examples/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/831 [00:00<?, ?it/s]

  0%|          | 0/53 [00:00<?, ?it/s]

...

{'eval_loss': 0.8452609777450562, 'eval_runtime': 4.6809, 'eval_samples_per_second': 180.307, 'eval_steps_per_second': 11.323, 'epoch': 2.0}


  0%|          | 0/53 [00:00<?, ?it/s]

{'eval_loss': 0.8412941098213196, 'eval_runtime': 4.6846, 'eval_samples_per_second': 180.165, 'eval_steps_per_second': 11.314, 'epoch': 3.0}
{'train_runtime': 331.0804, 'train_samples_per_second': 40.06, 'train_steps_per_second': 2.51, 'train_loss': 1.0283804295582317, 'epoch': 3.0}


  0%|          | 0/53 [00:00<?, ?it/s]

Balanced Accuracy: 0.38996600275515514


In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split

# Convert DataFrames to Datasets
train_dataset = Dataset.from_pandas(train_df_augmented)
val_dataset = Dataset.from_pandas(val_df)

# Combine into a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

# Tokenize the dataset
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)


training_args = TrainingArguments(
    output_dir="./roberta3aug",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)


# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

In [18]:
# Convert DataFrames to Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Combine into a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

# Tokenize the dataset
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)


training_args = TrainingArguments(
    output_dir="./roberta3aug",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)


# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

Map:   0%|          | 0/3374 [00:00<?, ? examples/s]

Map:   0%|          | 0/844 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/ikhlasul.hanif/miniconda3/envs/satdat/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  0%|          | 0/318 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

...

{'eval_loss': 0.8307819962501526, 'eval_runtime': 3.769, 'eval_samples_per_second': 223.931, 'eval_steps_per_second': 7.164, 'epoch': 2.0}


  0%|          | 0/27 [00:00<?, ?it/s]

{'eval_loss': 0.8319176435470581, 'eval_runtime': 4.2487, 'eval_samples_per_second': 198.649, 'eval_steps_per_second': 6.355, 'epoch': 3.0}
{'train_runtime': 188.3223, 'train_samples_per_second': 53.748, 'train_steps_per_second': 1.689, 'train_loss': 0.9069303116708431, 'epoch': 3.0}


  0%|          | 0/27 [00:00<?, ?it/s]

Balanced Accuracy: 0.3708772325647581
